# 03 · Retrieval evaluation and failure analysis

**Goal:** execute a tiny labeled benchmark, compare the two lexical methods, inspect failures and distinguish retrieval quality from the accuracy of a real recommendation. All relevance labels are synthetic.

## Step 1 — Load the versioned, synthetic judgments

In [ ]:
from savoroute.data.catalog import load_catalog
from savoroute.evaluation.retrieval import evaluate_search, load_benchmark

items = load_catalog().items
cases = load_benchmark()
print("Total cases:", len(cases))
print("Positive:", sum(bool(c.relevant_item_ids) for c in cases))
print("No-match:", sum(not c.relevant_item_ids for c in cases))

## Step 2 — Run reproducible ranking metrics

Recall@k measures the proportion of relevant items retrieved; MRR@k captures the rank of the first relevant hit; nDCG@k discounts lower-ranked hits. Empty-relevance cases have a separate no-match metric.

In [ ]:
k = 5
summaries = {method: evaluate_search(items, cases, at_k=k, method=method)
             for method in ("overlap", "bm25")}
for method, summary in summaries.items():
    print(method, summary.as_dict())

## Step 3 — Inspect missed queries

The synthetic paraphrase cases expose vocabulary mismatches not captured by aggregate scores.

In [ ]:
for method, summary in summaries.items():
    failures = [case for case in summary.cases
                if case.recall_at_k is not None and case.recall_at_k < 1]
    print("\n", method, "misses:", len(failures))
    for row in failures:
        print(row.case_id, repr(row.query), "->", row.retrieved_item_ids)

## Step 4 — No-match checks are not positive relevance judgments

In [ ]:
for row in summaries["bm25"].cases:
    if row.correct_empty is not None:
        print(row.case_id, "correctly empty:", row.correct_empty)
assert summaries["bm25"].empty_queries == 2

## Step 5 — Change k and compare

Try `at_k = 1`, `3`, or `10`. On this tiny dataset, near-identical values do not imply production-level performance.

In [ ]:
for at_k in (1, 3, 5):
    result = evaluate_search(items, cases, method="bm25", at_k=at_k)
    print(at_k, "Recall:", round(result.mean_recall_at_k, 3),
          "MRR:", round(result.mrr_at_k, 3))

The next stage adds source ingestion and structured extraction. No extraction adapter is included in this release.